### Kaczmarz algorithm

Iterative method to find the solution of square linear systems $A \vec{x} = \vec{b}$. The iterative step is defined as:

$$
    \vec{x_{k + 1}} = \vec{x_k} + \frac{b_j - \vec{a_j} \cdot \vec{x_k}}{\vec{a_j} \cdot \vec{a_j}}
$$

Where $\vec{a_j}$ is the j-th **row** of matrix $A$. For each step, the j-th row is choosen randomly with probability proportional to its squared norm

In [1]:
import numpy as np

In [2]:
def Kaczmarz(A, b, x0 = None, eps = 1e-04, max_iter = 1_000):
    A = np.asarray(A, dtype = float)
    b = np.asarray(b, dtype = float)
    N = A.shape[0]

    if x0 is None: # Random vector projected onto a random hyperplane a_i^T x = b_i
        i = np.random.randint(N)
        a_i = A[i, :]
        x = np.random.randn(N)
        x += (b[i] - np.dot(a_i, x)) / np.dot(a_i, a_i) * a_i
    else:
        x = np.asarray(x0, dtype = float).copy()

    # Sampling probabilities proportional to |a_i|^2
    row_norms_sq = np.sum(A ** 2, axis = 1)
    p = row_norms_sq / (np.linalg.norm(A, 'fro') ** 2)

    iterations = 0
    residual = b - A @ x
    res_norm = np.linalg.norm(residual)

    for _ in range(max_iter):
        if res_norm < eps: # Early stopping criterion active
            break

        i = np.random.choice(N, p = p)
        a_i = A[i, :]
        x += (b[i] - np.dot(a_i, x)) / row_norms_sq[i] * a_i

        iterations += 1
        residual = b - A @ x
        res_norm = np.linalg.norm(residual)
    else: # Loop finished without break
        if res_norm >= eps:
            print("Maximum number of iterations reached")

    stats = {
        "iterations": iterations,
        "residual": residual,
        "residual_norm": res_norm,
    }
    return x, stats

In [3]:
# Linear system A @ x = b
A = np.array([
    [ 2, -3,  5, -2],
    [ 4,  2, -3,  7],
    [-3,  5,  2, -3],
    [ 5, -1, -4,  2]
])

b = np.array([4, -1, 7, -3])

In [4]:
x, stats = Kaczmarz(A, b) # Approximated solution
x, stats

(array([ 0.63453794,  1.12984262,  1.07749374, -0.36647108]),
 {'iterations': 170,
  'residual': array([ 4.11451322e-05, -5.82427408e-05,  0.00000000e+00,  7.00227728e-05]),
  'residual_norm': np.float64(9.994162030079459e-05)})

In [5]:
err, err_stats = Kaczmarz(A, b - A @ x, eps = 1e-07) # Error estimation
err, err_stats

(array([ 1.70099193e-05,  2.98726962e-07, -6.81923281e-06, -2.10482243e-05]),
 {'iterations': 427,
  'residual': array([ 2.11899457e-08, -4.06575815e-20,  2.99156253e-08,  9.14208498e-08]),
  'residual_norm': np.float64(9.849736148356097e-08)})

In [6]:
np.linalg.solve(A, b) # Exact solution

array([ 0.63455497,  1.12984293,  1.07748691, -0.36649215])